In [2]:
from memoria_icm.datos import *
import numpy as np, pandas as pd, seaborn as sns
from memoria_icm.graficos import *
import matplotlib.pyplot as plt

In [3]:
import sqlite3
conn = sqlite3.connect(os.path.join(BASE_DIR, "data", "data_raw.db"))
df = pd.read_sql('SELECT * FROM datos_trackeo', conn)
conn.close()
columnas_basicas = ['id', 't', 'x', 'y']
metadatos = metadatos
columnas_a_mantener = columnas_basicas + metadatos + ['duracion_video']
df = df[columnas_a_mantener]

In [4]:
df

,id,t,x,y,id_ubicacion,especie,tratamiento,salinidad,nombre_archivo,estadio_larval,obs,video_analizado,larva_unica,duracion_video
0,0,0.000000,1.976667,-0.624416,1,CC,E,35-N,Video_001,Larva recién eclosionada,0,1,1,11.88146
1,0,0.073174,1.981245,-0.629966,1,CC,E,35-N,Video_001,Larva recién eclosionada,0,1,1,11.88146
2,0,0.123811,1.972938,-0.639559,1,CC,E,35-N,Video_001,Larva recién eclosionada,0,1,1,11.88146
3,0,0.181472,1.974790,-0.639965,1,CC,E,35-N,Video_001,Larva recién eclosionada,0,1,1,11.88146
4,0,0.224938,1.972984,-0.641756,1,CC,E,35-N,Video_001,Larva recién eclosionada,0,1,1,11.88146
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95619,320,15.362240,2.660554,-3.178143,3,LA,R,35,Video_006,Larva equinopluteus,0,1,1,15.56905
95620,320,15.418690,2.656156,-3.176854,3,LA,R,35,Video_006,Larva equinopluteus,0,1,1,15.56905
95621,320,15.463080,2.651592,-3.176066,3,LA,R,35,Video_006,Larva equinopluteus,0,1,1,15.56905
95622,320,15.510260,2.649979,-3.176235,3,LA,R,35,Video_006,Larva equinopluteus,0,1,1,15.56905


In [5]:
dataframe = definir_dtypes(df)
dataframe = calcular_coordenadas_centradas(dataframe)
dataframe = agregar_datos_calculados(dataframe)

In [7]:
def resumir_por_id(df, columnas_a_mantener, variables_base):
    df = df.copy()
    estadisticas = ('mean', 'median')
    sufijos = {'mean': 'promedio', 'median': 'mediana'}

    # Parte 1: resumen estadístico
    df_resumen = df.groupby('id')[variables_base].agg(estadisticas)
    df_resumen.columns = [f"{col}_{sufijos.get(stat, stat)}" for col, stat in df_resumen.columns]
    df_resumen = df_resumen.apply(pd.to_numeric, errors='coerce')
    df_resumen.reset_index(inplace=True)

    # Parte 2: metadatos y columnas a mantener
    columnas_extra = [col for col in columnas_a_mantener if col not in ['t', 'x', 'y']]
    df_metadata = df[columnas_extra].drop_duplicates(subset='id')

    # Unir
    df_final = pd.merge(df_metadata, df_resumen, on='id', how='inner')

    # Ordenar columnas: id, luego metadatos, luego resumenes
    columnas_finales = (
        ['id'] +
        [col for col in columnas_extra if col != 'id'] +
        [col for col in df_final.columns if col not in columnas_extra + ['id']]
    )
    df_final = df_final[columnas_finales]

    return df_final


In [8]:
resumen = resumir_por_id(dataframe, columnas_a_mantener, variables_base)
resumen

,id,id_ubicacion,especie,tratamiento,salinidad,nombre_archivo,estadio_larval,obs,video_analizado,larva_unica,...,a_angular_centrado_promedio,a_angular_centrado_mediana,curvatura_promedio,curvatura_mediana,curvatura_centrada_promedio,curvatura_centrada_mediana,theta_v_promedio,theta_v_mediana,theta_v_centrado_promedio,theta_v_centrado_mediana
0,0,1,CC,E,35-N,Video_001,Larva recién eclosionada,0,1,1,...,1.132029,-0.109464,85.433945,65.455025,85.433945,65.455025,0.516923,0.653789,0.516923,0.653789
1,1,1,CC,E,35-N,Video_002,Larva recién eclosionada,0,1,1,...,-2.971971,-0.081397,234.026472,89.326773,234.026472,89.326773,0.015951,-0.083172,0.015951,-0.083172
2,2,1,CC,E,35-N,Video_003,Larva recién eclosionada,0,0,1,...,4.440467,0.005032,34.483809,19.023583,34.483809,19.023583,1.439958,2.661705,1.561961,2.674525
3,3,1,CC,E,35-N,Video_004,Larva recién eclosionada,0,1,1,...,-3.756175,-0.098247,230.251134,188.081244,230.251134,188.081244,0.473841,0.938891,0.473841,0.938891
4,4,1,CC,E,35-N,Video_005,Larva recién eclosionada,1,1,1,...,4.141833,0.391242,208.520365,142.603931,208.520365,142.603931,0.232893,0.449934,0.228781,0.410176
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
300,316,3,LA,R,35,Video_002,Larva equinopluteus,0,1,0,...,0.600819,0.021219,34.370852,21.977657,34.370852,21.977657,0.875457,2.240607,0.875457,2.240607
301,317,3,LA,R,35,Video_003,Larva equinopluteus,0,1,1,...,-5.055701,0.020641,89.862250,56.157837,89.862250,56.157837,1.759692,1.936857,1.759692,1.936857
302,318,3,LA,R,35,Video_004,Larva equinopluteus,0,1,1,...,1.902002,0.004635,161.290648,21.684518,161.290648,21.684518,-0.201713,-0.289052,-0.201713,-0.289052
303,319,3,LA,R,35,Video_005,Larva equinopluteus,0,1,1,...,4.178734,0.099680,105.718632,63.501870,105.718632,63.501870,0.332030,0.073811,0.332030,0.073811


In [9]:
dataframe = dataframe[~((dataframe['salinidad'].isin(['05', '10', '15'])) & (dataframe['especie'] == 'CC'))]
resumen = resumen[~((resumen['salinidad'].isin(['05', '10', '15'])) & (resumen['especie'] == 'CC'))]

In [10]:
umbral_duracion = 11.515
df_resumen_filtrado = resumen[resumen["duracion_video"] >= umbral_duracion].copy()
ids_validos = df_resumen_filtrado["id"].unique()

df_filtrado = dataframe[dataframe["id"].isin(ids_validos) & (dataframe["t"] <= umbral_duracion)].copy()
df_filtrado.loc[:, "duracion_video"] = df_filtrado["id"].map(resumen.set_index("id")["duracion_video"])
df_filtrado.loc[:, "duracion_video_recortada"] = umbral_duracion

In [11]:
dataframe = df_filtrado
resumen = resumir_por_id(dataframe, columnas_a_mantener, variables_base)

In [11]:
resumen

,id,id_ubicacion,especie,tratamiento,salinidad,nombre_archivo,estadio_larval,obs,video_analizado,larva_unica,...,a_angular_centrado_promedio,a_angular_centrado_mediana,curvatura_promedio,curvatura_mediana,curvatura_centrada_promedio,curvatura_centrada_mediana,theta_v_promedio,theta_v_mediana,theta_v_centrado_promedio,theta_v_centrado_mediana
0,0,1,CC,E,35-N,Video_001,Larva recién eclosionada,0,1,1,...,1.166878,-0.107830,86.111707,67.287227,86.111707,67.287227,0.446752,0.505336,0.446752,0.505336
1,1,1,CC,E,35-N,Video_002,Larva recién eclosionada,0,1,1,...,-3.061171,-0.102260,238.659192,94.229832,238.659192,94.229832,0.015796,-0.120120,0.015796,-0.120120
2,3,1,CC,E,35-N,Video_004,Larva recién eclosionada,0,1,1,...,-3.860114,-0.027329,229.644998,183.808560,229.644998,183.808560,0.478789,0.926164,0.478789,0.926164
3,4,1,CC,E,35-N,Video_005,Larva recién eclosionada,1,1,1,...,4.159481,0.391242,211.802054,142.603931,211.802054,142.603931,0.210658,0.276620,0.206403,0.260250
4,5,1,CC,E,35-N,Video_006,Larva recién eclosionada,0,1,1,...,-3.250676,-0.033807,166.512069,36.641630,166.512069,36.641630,-0.719150,-1.145635,-0.719150,-1.145635
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,316,3,LA,R,35,Video_002,Larva equinopluteus,0,1,0,...,0.805636,0.019622,36.039624,23.368477,36.039624,23.368477,2.067259,2.437282,2.067259,2.437282
262,317,3,LA,R,35,Video_003,Larva equinopluteus,0,1,1,...,-6.663943,-0.000412,112.075803,86.970219,112.075803,86.970219,1.743853,2.130771,1.743853,2.130771
263,318,3,LA,R,35,Video_004,Larva equinopluteus,0,1,1,...,2.595045,0.010800,23.874776,14.785156,23.874776,14.785156,-0.467722,-0.365288,-0.467722,-0.365288
264,319,3,LA,R,35,Video_005,Larva equinopluteus,0,1,1,...,5.702312,0.143047,85.503557,49.689911,85.503557,49.689911,0.482970,0.186716,0.482970,0.186716


In [12]:
resumen.drop(columns = ['v_centrado_promedio', 'v_centrado_mediana', 'a_centrado_promedio', 'a_centrado_mediana', 'theta_v_centrado_promedio', 'theta_v_centrado_mediana', 'curvatura_centrada_promedio', 'curvatura_centrada_mediana'], inplace = True)
dataframe.drop(columns = ['v_centrado', 'a_centrado', 'theta_v_centrado', 'curvatura_centrada'], inplace = True)

In [13]:
dataframe["nivel_salinidad"] = dataframe["salinidad"].apply(clasificar_salinidad).astype('category')
resumen["nivel_salinidad"] = resumen["salinidad"].apply(clasificar_salinidad).astype('category')

In [14]:
dataframe

,id,t,x,y,id_ubicacion,especie,tratamiento,salinidad,nombre_archivo,estadio_larval,...,theta_r,theta_r_centrado,v_angular,v_angular_centrado,a_angular,a_angular_centrado,theta_v,curvatura,duracion_video_recortada,nivel_salinidad
0,0,0.000000,1.976667,-0.624416,1,CC,E,35-N,Video_001,Larva recién eclosionada,...,-0.305973,0.000000,-0.025723,-12.040794,-0.688948,-51.274027,-0.881077,0.000000,11.515,normal
1,0,0.073174,1.981245,-0.629966,1,CC,E,35-N,Video_001,Larva recién eclosionada,...,-0.307856,-0.881077,-0.076137,-15.792735,-0.074625,60.562228,-2.033736,126.445970,11.515,normal
2,0,0.123811,1.972938,-0.639559,1,CC,E,35-N,Video_001,Larva recién eclosionada,...,-0.313477,-1.812237,-0.058389,-8.807270,0.549592,142.715592,-2.177735,147.635591,11.515,normal
3,0,0.181472,1.974790,-0.639965,1,CC,E,35-N,Video_001,Larva recién eclosionada,...,-0.313388,-1.690928,-0.013625,-0.265367,0.178689,107.160874,-1.927719,764.513321,11.515,normal
4,0,0.224938,1.972984,-0.641756,1,CC,E,35-N,Video_001,Larva recién eclosionada,...,-0.314478,-1.780087,-0.025440,3.049804,-0.242878,56.515627,-0.930846,305.291105,11.515,normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95538,320,11.291740,2.965504,-3.120556,3,LA,R,35,Video_006,Larva equinopluteus,...,-0.810869,-2.648078,-0.029061,0.006991,-0.050172,0.096506,-2.552953,16.924950,11.515,normal
95539,320,11.348250,2.959012,-3.125446,3,LA,R,35,Video_006,Larva equinopluteus,...,-0.812745,-2.647365,-0.025069,0.021723,-0.157602,0.140003,-2.293583,78.380508,11.515,normal
95540,320,11.393450,2.957334,-3.128929,3,LA,R,35,Video_006,Larva equinopluteus,...,-0.813585,-2.646054,-0.040444,0.023688,-0.254000,-0.067894,-2.407994,75.171379,11.515,normal
95541,320,11.429330,2.950067,-3.134238,3,LA,R,35,Video_006,Larva equinopluteus,...,-0.815659,-2.645356,-0.047104,0.018080,0.010553,-0.110120,-2.490801,11.765173,11.515,normal


In [ ]:
# import sqlite3
# conn = sqlite3.connect(os.path.join(BASE_DIR, "data", "data.db"))
# dataframe.to_sql('datos_trackeo', conn, if_exists = 'replace')
# resumen.to_sql('datos_resumen', conn, if_exists = 'replace')
# conn.close()

In [33]:
#falta solo evaluar si eliminar larva prisma o no